# 01 - Começando Camada Bronze — Histórico

Instalando yFinance para consulta e download de dados históricos

---

In [0]:
%pip install yfinance
dbutils.library.restartPython()

---

Importando todas as bibliotecas que serão utilizadas durante o notebook, em seguida definimos todas as variáveis que serão utilizadas durante a execução.

# .
### Variáveis e suas utilizações
bronze_path = define o caminho onde será criada a tabela no Unity Catalog.

tickers = informa os tickers que serão buscados.

periodo = informa o período de busca em relação à data atual.

intervalo = informa o intervalo de tempo de cada variação da ação.

---

In [0]:
import yfinance as yf
from pyspark.sql.functions import to_timestamp, col, current_timestamp
import pandas as pd

bronze_path = "workspace.stocks.bronze"
tickers = ["PETR4.SA", "VALE3.SA", "BBAS3.SA"]

periodo = "1y"
intervalo = "1d"


---

Função responsável por baixar o histórico de cada ação em loop, resetamos o índice para o datetime voltar como coluna e não como índice, achatamos o MultiIndex das colunas e padronizamos os nomes em minúsculo. Cada dataframe gerado é salvo em um array e ao final concatenamos todos em um único dataframe pandas.

---

In [0]:
def download_stocks(tickers: list, periodo: str, intervalo: str):
        tickers_hist = []

        for t in tickers:
                print(f"-----Buscando histórico de {t}-----")
                try:
                        stock = yf.download(t,
                                        period= periodo,
                                        interval= intervalo,
                                        progress=False
                        )

                        stock = stock.reset_index()
                        stock.columns = stock.columns.get_level_values(0)
                
                        stock["ticker"] = t
                        stock["fonte"] = "historico"
                        tickers_hist.append(stock)
                        print(f"-----Histórico de {t} baixado com sucesso-----")
                        
                except Exception as e:
                        print(f"-----Histórico de {t} não encontrado-----")
                        print(e)

        tickers_hist_df = pd.concat(tickers_hist, ignore_index=True)
        tickers_hist_df.columns = [c.lower() for c in tickers_hist_df.columns]

        return tickers_hist_df

print(f"-----Baixando Histórico de {len(tickers)} ações-----")

hist_df = download_stocks(tickers, periodo, intervalo)

print(f"-----Encontrado {len(hist_df)} registros | Ações: {hist_df['ticker'].unique()}-----")


---

Convertendo o dataframe pandas para Spark, realizamos os casts das colunas para os tipos corretos, adicionamos a coluna de controle de ingestão e salvamos na tabela bronze no formato Delta com particionamento por ticker.

---

In [0]:
df = spark.createDataFrame(hist_df)

df = (df.withColumn("event_time", to_timestamp(col("date")))
           .withColumn("ingestao_ts", current_timestamp())
           .withColumn("open", col("open").cast("double"))
           .withColumn("high", col("high").cast("double"))
           .withColumn("low", col("low").cast("double"))
           .withColumn("close", col("close").cast("double"))
           .withColumn("volume", col("volume").cast("long"))
           .drop("date")
        ).select(
                "ticker", "event_time","open", "high", "low", "close", "volume","fonte", "ingestao_ts"
        )
(
    df.write.format("delta")
    .mode("overwrite")
    .partitionBy("ticker")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_path)
)

df.printSchema()